#### 1. Import Required Libraries
The following Python libraries are used for data manipulation, statistical analysis, visualization, and exploratory data analysis.

In [1]:
# Data manipulation
import pandas as pd
import numpy as np

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Statistical analysis
from scipy import stats

#### 2. Data Loading

The analysis uses agricultural commodity price data obtained from two different sources. The first dataset contains the primary market price observations used in the initial AgriPulse analysis, while the second dataset from FEWS NET provides additional commodity and price observations that may help address gaps in the primary dataset.

The datasets will first be loaded separately to allow for independent inspection and cleaning before determining the appropriate approach for integrating them.

In [2]:
fews_df = pd.read_csv("Completed forecasting dataset with rainfall.csv")
fews_df.head()

,commodity,market,pricetype,date,price_per_kg,rainfall_mm,GPR,diesel_price_kes,source,admin1,latitude,longitude,category,rainfall_source
0,Beans (dry),Busia,Wholesale,2014-11-01,75.500000,113.30,NaN,NaN,FEWS,Western,0.462451,34.1065,pulses and nuts,NASA POWER PRECTOTCORR monthly sum
1,Beans (dry),Busia,Wholesale,2016-03-01,95.000000,65.93,NaN,NaN,FEWS,Western,0.462451,34.1065,pulses and nuts,NASA POWER PRECTOTCORR monthly sum
2,Beans (dry),Busia,Wholesale,2016-04-01,90.000000,264.59,NaN,NaN,FEWS,Western,0.462451,34.1065,pulses and nuts,NASA POWER PRECTOTCORR monthly sum
3,Beans (dry),Busia,Wholesale,2016-05-01,100.000000,208.40,NaN,NaN,FEWS,Western,0.462451,34.1065,pulses and nuts,NASA POWER PRECTOTCORR monthly sum
4,Beans (dry),Busia,Wholesale,2016-06-01,87.222222,84.67,NaN,NaN,FEWS,Western,0.462451,34.1065,pulses and nuts,NASA POWER PRECTOTCORR monthly sum


In [78]:
fews_df.describe

<bound method NDFrame.describe of               commodity  market  pricetype       date  price_per_kg  \
0           Beans (dry)   Busia  Wholesale 2014-11-01     75.500000   
1           Beans (dry)   Busia  Wholesale 2016-03-01     95.000000   
2           Beans (dry)   Busia  Wholesale 2016-04-01     90.000000   
3           Beans (dry)   Busia  Wholesale 2016-05-01    100.000000   
4           Beans (dry)   Busia  Wholesale 2016-06-01     87.222222   
...                 ...     ...        ...        ...           ...   
22044  Potatoes (Irish)  Nakuru  Wholesale 2019-11-15     54.900000   
22045  Potatoes (Irish)  Nakuru  Wholesale 2019-12-15     52.950000   
22046  Potatoes (Irish)  Nakuru  Wholesale 2020-01-15     50.000000   
22047  Potatoes (Irish)  Nakuru  Wholesale 2020-02-15     50.000000   
22048  Potatoes (Irish)  Nakuru  Wholesale 2020-03-15     52.500000   

       rainfall_mm         GPR  diesel_price_kes source       admin1  \
0       113.300000         NaN           

In [3]:
agri_df = pd.read_csv("agripulse_agricultural_produce.csv")
agri_df.head()

,date,region,county,market,market_id,latitude,longitude,category,commodity,commodity_id,...,lag_1,lag_3,lag_6,lag_12,rolling_3m_avg,rolling_6m_avg,rolling_12m_avg,price_vs_regional_avg,price_label,commodity_type
0,2024-09-15,Coast,Tana River,Adele Center,10441,-0.47,39.61,pulses and nuts,Beans,50,...,NaN,NaN,NaN,NaN,160.000000,160.000000,160.000000,1.050687,Average,Legume
1,2024-03-15,North Eastern,Garissa,Alango Arba,10354,-0.10,39.99,pulses and nuts,Beans,50,...,NaN,NaN,NaN,NaN,200.000000,200.000000,200.000000,1.097394,Average,Legume
2,2024-06-15,North Eastern,Garissa,Alango Arba,10354,-0.10,39.99,pulses and nuts,Beans,50,...,200.0,NaN,NaN,NaN,200.000000,200.000000,200.000000,1.201201,Expensive,Legume
3,2024-09-15,North Eastern,Garissa,Alango Arba,10354,-0.10,39.99,pulses and nuts,Beans,50,...,200.0,NaN,NaN,NaN,196.666667,196.666667,196.666667,1.143326,Expensive,Legume
4,2023-12-15,Rift Valley,Turkana,Alemsekon,10457,3.53,34.82,pulses and nuts,Beans,50,...,NaN,NaN,NaN,NaN,155.000000,155.000000,155.000000,0.888363,Cheap,Legume


In [4]:
fews_df.columns.tolist()

['commodity',
 'market',
 'pricetype',
 'date',
 'price_per_kg',
 'rainfall_mm',
 'GPR',
 'diesel_price_kes',
 'source',
 'admin1',
 'latitude',
 'longitude',
 'category',
 'rainfall_source']

In [5]:
agri_df.columns.tolist()

['date',
 'region',
 'county',
 'market',
 'market_id',
 'latitude',
 'longitude',
 'category',
 'commodity',
 'commodity_id',
 'unit',
 'priceflag',
 'pricetype',
 'currency',
 'price',
 'usdprice',
 'date_month',
 'rainfall_mm',
 'price_per_kg',
 'month',
 'year',
 'season',
 'lag_1',
 'lag_3',
 'lag_6',
 'lag_12',
 'rolling_3m_avg',
 'rolling_6m_avg',
 'rolling_12m_avg',
 'price_vs_regional_avg',
 'price_label',
 'commodity_type']

In [6]:
fews_df["date"] = pd.to_datetime(fews_df["date"])
fews_df["date"].dtype

dtype('<M8[ns]')

In [7]:
agri_df["date"] = pd.to_datetime(agri_df["date"])
agri_df["date"].dtype

dtype('<M8[ns]')

In [8]:
fews_new = fews_df.drop(['GPR','diesel_price_kes','source','rainfall_source'], axis =1)

In [9]:
fews_new.head()

,commodity,market,pricetype,date,price_per_kg,rainfall_mm,admin1,latitude,longitude,category
0,Beans (dry),Busia,Wholesale,2014-11-01,75.500000,113.30,Western,0.462451,34.1065,pulses and nuts
1,Beans (dry),Busia,Wholesale,2016-03-01,95.000000,65.93,Western,0.462451,34.1065,pulses and nuts
2,Beans (dry),Busia,Wholesale,2016-04-01,90.000000,264.59,Western,0.462451,34.1065,pulses and nuts
3,Beans (dry),Busia,Wholesale,2016-05-01,100.000000,208.40,Western,0.462451,34.1065,pulses and nuts
4,Beans (dry),Busia,Wholesale,2016-06-01,87.222222,84.67,Western,0.462451,34.1065,pulses and nuts


In [11]:
agri_df = agri_df[agri_df["commodity"] != "Rice (imported, Pakistan)"
].copy()

In [12]:
# Check whether the unwanted commodity is still present
print("Rice (imported, Pakistan)" in agri_df["commodity"].unique())

False


In [13]:
# Define the columns that identify a unique price observation
key_cols = ["commodity", "market", "pricetype", "date"]

In [14]:
# Define the columns that identify one monthly commodity observation.
# We use date_month instead of the original date because AgriPulse
# and FEWS use different days within the same month.

monthly_key = [
    "commodity",
    "market",
    "pricetype",
    "date_month"
]

In [15]:
# Check for duplicate observations using the proposed key

print("Duplicates in agri_df:", agri_df.duplicated(subset=key_cols).sum())
print("Duplicates in fews_new:", fews_new.duplicated(subset=key_cols).sum())

Duplicates in agri_df: 0
Duplicates in fews_new: 0


In [17]:
# Find observations that exist in FEWS but not in AgriPulse

fews_only = fews_new.merge(
    agri_df[key_cols].drop_duplicates(),
    on=key_cols,
    how="left",
    indicator=True
)

fews_only = fews_only[
    fews_only["_merge"] == "left_only"
].copy()

In [18]:
print("FEWS-only observations:", fews_only.shape[0])

FEWS-only observations: 20985


In [19]:
fews_only[key_cols].head(10)

,commodity,market,pricetype,date
0,Beans (dry),Busia,Wholesale,2014-11-01
1,Beans (dry),Busia,Wholesale,2016-03-01
2,Beans (dry),Busia,Wholesale,2016-04-01
3,Beans (dry),Busia,Wholesale,2016-05-01
4,Beans (dry),Busia,Wholesale,2016-06-01
5,Beans (dry),Busia,Wholesale,2016-07-01
6,Beans (dry),Busia,Wholesale,2016-08-01
7,Beans (dry),Busia,Wholesale,2016-09-01
8,Beans (dry),Busia,Wholesale,2016-10-01
9,Beans (dry),Busia,Wholesale,2016-11-01


In [20]:
print("Percentage of FEWS observations not found in AgriPulse:",
      round(len(fews_only) / len(fews_df) * 100, 2), "%")

Percentage of FEWS observations not found in AgriPulse: 95.17 %


In [21]:
# Checking the commodities fews will fill to fix the gap
fews_only["commodity"].value_counts()

commodity
Maize               11592
Beans (mixed)        6522
Beans (dry)          1324
Sorghum              1193
Maize meal            231
Potatoes (Irish)      123
Name: count, dtype: int64

In [22]:
# checking the years being filled
fews_only["year"] = fews_only["date"].dt.year

In [23]:
fews_only["year"].value_counts().sort_index()

year
2006     199
2007     207
2008     484
2009     485
2010     470
2011     510
2012     520
2013     565
2014     535
2015     545
2016     682
2017     652
2018    1804
2019    1765
2020    1404
2021    1790
2022    1596
2023    1991
2024    1932
2025    1781
2026    1068
Name: count, dtype: int64

In [24]:
commodity_year_coverage = (
    fews_only
    .groupby(["commodity", "year"])
    .size()
    .reset_index(name="observations")
    .sort_values(["commodity", "year"])
)

display(commodity_year_coverage.head(20))

,commodity,year,observations
0,Beans (dry),2006,48
1,Beans (dry),2007,48
2,Beans (dry),2008,51
3,Beans (dry),2009,51
4,Beans (dry),2010,49
5,Beans (dry),2011,65
6,Beans (dry),2012,69
7,Beans (dry),2013,77
8,Beans (dry),2014,66
9,Beans (dry),2015,67


In [25]:
# markets being filled
fews_only["market"].value_counts().head(20)

market
Nairobi         912
Eldoret         910
Kitui           887
Kisumu          659
Mombasa         593
Kitale          477
Isiolo          388
Taita Taveta    380
Kilifi          380
Garissa         360
Meru            355
Makueni         344
Embu-Mbeere     344
Nyeri-Kieni     344
Lamu            343
Kajiado         302
Kwale           299
Marsabit        292
Busia           241
Baringo         220
Name: count, dtype: int64

In [26]:
fews_only["admin1"].value_counts().head(20)

admin1
Rift Valley      4672
Eastern          4499
Coast            3300
Central          2230
Nairobi          1833
Nyanza           1829
North Eastern    1355
Western          1161
Name: count, dtype: int64

In [27]:
# Check the earliest and latest dates available in each dataset
# This helps us understand how much historical coverage FEWS adds
print("AgriPulse date range:")
print(agri_df["date"].min(), "to", agri_df["date"].max())

print("\nFEWS date range:")
print(fews_df["date"].min(), "to", fews_df["date"].max())

AgriPulse date range:
2006-01-15 00:00:00 to 2026-08-15 00:00:00

FEWS date range:
2006-01-01 00:00:00 to 2026-08-15 00:00:00


In [28]:
# Convert the FEWS date column to datetime format.
# This is necessary before using datetime operations such as .dt.day.

fews_df["date"] = pd.to_datetime(fews_df["date"])

In [31]:
# Count the number of unique dates in each dataset
# This helps us compare how frequently each source records observations
print("AgriPulse unique dates:", agri_df["date"].nunique())
print("FEWS unique dates:", fews_df["date"].nunique())

AgriPulse unique dates: 248
FEWS unique dates: 551


In [32]:
# Check how dates are represented within each dataset.
# This helps us determine whether one source records monthly observations
# on the 1st of the month while the other uses another day, such as the 15th.

print("AgriPulse day-of-month distribution:")
display(agri_df["date"].dt.day.value_counts().sort_index())

print("\nFEWS day-of-month distribution:")
display(fews_df["date"].dt.day.value_counts().sort_index())

AgriPulse day-of-month distribution:


date
15    16327
Name: count, dtype: int64


FEWS day-of-month distribution:


date
1     20787
15     1187
28        4
29        2
30       25
31       44
Name: count, dtype: int64

In [33]:
# Create a month-level date key for both datasets.
# This converts dates such as 2016-03-01 and 2016-03-15
# into the same monthly timestamp: 2016-03-01.
#
# We keep the original 'date' column unchanged because it may
# still be useful for tracing observations back to their source.

agri_df["date_month"] = agri_df["date"].dt.to_period("M")
fews_df["date_month"] = fews_df["date"].dt.to_period("M")

In [34]:
# Display a few original dates alongside their new month-level keys.
# This lets us confirm that different days within the same month
# are now represented consistently.

print("AgriPulse:")
display(agri_df[["date", "date_month"]].head())

print("\nFEWS:")
display(fews_df[["date", "date_month"]].head())

AgriPulse:


,date,date_month
0,2024-09-15,2024-09
1,2024-03-15,2024-03
2,2024-06-15,2024-06
3,2024-09-15,2024-09
4,2023-12-15,2023-12



FEWS:


,date,date_month
0,2014-11-01,2014-11
1,2016-03-01,2016-03
2,2016-04-01,2016-04
3,2016-05-01,2016-05
4,2016-06-01,2016-06


In [35]:
# Define the columns we will use to identify a monthly observation.
# We use date_month instead of the original date so that different
# days within the same month are treated as the same period.

monthly_key = ["commodity", "market", "pricetype", "date_month"]

In [36]:
# Check whether either dataset contains more than one observation
# for the same commodity, market, price type, and month.
# This is important before using these columns as our matching key.

print("AgriPulse monthly duplicates:")
print(agri_df.duplicated(subset=monthly_key).sum())

print("\nFEWS monthly duplicates:")
print(fews_df.duplicated(subset=monthly_key).sum())

AgriPulse monthly duplicates:
0

FEWS monthly duplicates:
0


In [37]:
# Identify FEWS observations that are not already present in AgriPulse
# using the standardized monthly matching key.
fews_only_monthly = fews_df.merge(
    agri_df[monthly_key].drop_duplicates(),
    on=monthly_key,
    how="left",
    indicator=True
)

# Keep only observations that exist in FEWS but not in AgriPulse
fews_only_monthly = fews_only_monthly[
    fews_only_monthly["_merge"] == "left_only"
].copy()

# Calculate how much of the FEWS dataset is additional coverage
fews_only_percentage = (
    len(fews_only_monthly) / len(fews_df)
) * 100

print("FEWS-only monthly observations:", len(fews_only_monthly))
print(
    "Percentage of FEWS observations not found in AgriPulse:",
    round(fews_only_percentage, 2),
    "%"
)

FEWS-only monthly observations: 19556
Percentage of FEWS observations not found in AgriPulse: 88.69 %


In [38]:
# Count the FEWS-only observations by commodity.
# This helps us see which commodities have additional coverage
# that is missing from AgriPulse.

fews_only_by_commodity = (
    fews_only_monthly["commodity"]
    .value_counts()
    .reset_index()
)

# Rename the columns to make the output easier to interpret
fews_only_by_commodity.columns = ["commodity", "fews_only_count"]

# Display the top 20 commodities with additional FEWS coverage
print(fews_only_by_commodity.head(20))

          commodity  fews_only_count
0             Maize            11106
1     Beans (mixed)             6522
2       Beans (dry)              853
3           Sorghum              721
4        Maize meal              231
5  Potatoes (Irish)              123


In [39]:
# Get the unique commodity names from each dataset
# so we can identify naming differences before standardizing them.

agri_commodities = set(agri_df["commodity"].dropna().unique())
fews_commodities = set(fews_df["commodity"].dropna().unique())

# Display all commodity names in AgriPulse
print("AgriPulse commodities:")
print(sorted(agri_commodities))

print("\nFEWS commodities:")
print(sorted(fews_commodities))

AgriPulse commodities:
['Beans', 'Beans (dolichos)', 'Beans (dry)', 'Beans (kidney)', 'Beans (mung)', 'Beans (rosecoco)', 'Beans (yellow)', 'Cabbage', 'Cowpea leaves', 'Cowpeas', 'Cowpeas (dry)', 'Kale', 'Maize', 'Maize (white)', 'Maize (white, dry)', 'Millet (finger)', 'Onions (dry)', 'Onions (red)', 'Pigeon peas (dry)', 'Potatoes (Irish)', 'Potatoes (Irish, red)', 'Potatoes (Irish, white)', 'Rice', 'Rice (aromatic)', 'Sorghum', 'Sorghum (red)', 'Sorghum (white)', 'Spinach', 'Tomatoes']

FEWS commodities:
['Beans (dry)', 'Beans (mixed)', 'Maize', 'Maize meal', 'Potatoes (Irish)', 'Sorghum']


In [56]:
# Convert the FEWS date column to datetime format.
# This ensures pandas can correctly extract the month.

fews_new["date"] = pd.to_datetime(fews_new["date"])

# Create a month-level date for matching FEWS observations
# with AgriPulse observations.

fews_new["date_month"] = fews_new["date"].dt.to_period("M")

# Confirm that the column was created correctly.
print(fews_new[["date", "date_month"]].head())

        date date_month
0 2014-11-01    2014-11
1 2016-03-01    2016-03
2 2016-04-01    2016-04
3 2016-05-01    2016-05
4 2016-06-01    2016-06


In [79]:
# This helps us confirm which varieties should be consolidated.

sorted(agric_df["commodity"].dropna().unique())

['Beans',
 'Beans (dolichos)',
 'Beans (dry)',
 'Beans (kidney)',
 'Beans (mixed)',
 'Beans (mung)',
 'Beans (rosecoco)',
 'Beans (yellow)',
 'Cabbage',
 'Cowpea leaves',
 'Cowpeas',
 'Cowpeas (dry)',
 'Kale',
 'Maize',
 'Maize (white)',
 'Maize (white, dry)',
 'Maize meal',
 'Millet (finger)',
 'Onions (dry)',
 'Onions (red)',
 'Pigeon peas (dry)',
 'Potatoes (Irish)',
 'Potatoes (Irish, red)',
 'Potatoes (Irish, white)',
 'Rice',
 'Rice (aromatic)',
 'Sorghum',
 'Sorghum (red)',
 'Sorghum (white)',
 'Spinach',
 'Tomatoes']

In [90]:
# Define the commodity varieties that should be consolidated.
# These are grouped because they represent varieties/forms
# of the same underlying commodity.

commodity_groups = {
    "Beans": [
        "Beans",
        "Beans (dolichos)",
        "Beans (dry)",
        "Beans (kidney)",
        "Beans (mung)",
        "Beans (rosecoco)",
        "Beans (yellow)",
        "Beans (mixed)"
    ],

    "Maize": [
        "Maize",
        "Maize (white)",
        "Maize (white, dry)"
    ],

    "Potatoes": [
        "Potatoes (Irish)",
        "Potatoes (Irish, red)",
        "Potatoes (Irish, white)"
    ],

    "Sorghum": [
        "Sorghum",
        "Sorghum (red)",
        "Sorghum (white)"
    ],

    "Onions": [
        "Onions (dry)",
        "Onions (red)"
    ]
}

# Display the consolidation groups for verification.
for standardized_name, varieties in commodity_groups.items():
    print(f"{standardized_name}:")
    print(varieties)
    print()

Beans:
['Beans', 'Beans (dolichos)', 'Beans (dry)', 'Beans (kidney)', 'Beans (mung)', 'Beans (rosecoco)', 'Beans (yellow)', 'Beans (mixed)']

Maize:
['Maize', 'Maize (white)', 'Maize (white, dry)']

Potatoes:
['Potatoes (Irish)', 'Potatoes (Irish, red)', 'Potatoes (Irish, white)']

Sorghum:
['Sorghum', 'Sorghum (red)', 'Sorghum (white)']

Onions:
['Onions (dry)', 'Onions (red)']



In [91]:
# Create a copy of the commodity column that will contain
# the consolidated commodity names.
# The original "commodity" column remains unchanged for now.

agri_df["commodity_consolidated"] = agri_df["commodity"]

# Replace the selected commodity varieties with their
# consolidated commodity names.

for standardized_name, varieties in commodity_groups.items():
    agri_df.loc[
        agri_df["commodity"].isin(varieties),
        "commodity_consolidated"
    ] = standardized_name

# Check the original and consolidated names side by side.
print(
    agri_df[
        ["commodity", "commodity_consolidated"]
    ].drop_duplicates().sort_values(
        ["commodity_consolidated", "commodity"]
    ).to_string(index=False)
)

              commodity commodity_consolidated
                  Beans                  Beans
       Beans (dolichos)                  Beans
            Beans (dry)                  Beans
         Beans (kidney)                  Beans
           Beans (mung)                  Beans
       Beans (rosecoco)                  Beans
         Beans (yellow)                  Beans
                Cabbage                Cabbage
          Cowpea leaves          Cowpea leaves
                Cowpeas                Cowpeas
          Cowpeas (dry)          Cowpeas (dry)
                   Kale                   Kale
                  Maize                  Maize
          Maize (white)                  Maize
     Maize (white, dry)                  Maize
        Millet (finger)        Millet (finger)
           Onions (dry)                 Onions
           Onions (red)                 Onions
      Pigeon peas (dry)      Pigeon peas (dry)
       Potatoes (Irish)               Potatoes
  Potatoes (I

In [92]:
# Identify the groups where multiple commodity varieties occur
# in the same market, price type, and month.
#
# These are the groups where consolidation will actually
# combine observations and require a median price.

consolidation_key = [
    "commodity_consolidated",
    "market",
    "pricetype",
    "date_month"
]

consolidation_check = (
    agri_df
    .groupby(consolidation_key)
    .size()
    .reset_index(name="number_of_rows")
)

# Show only groups containing more than one observation.
# These are the groups that will be affected by aggregation.

affected_groups = consolidation_check[
    consolidation_check["number_of_rows"] > 1
].copy()

print("Total groups:", len(consolidation_check))
print("Groups with multiple observations:", len(affected_groups))
print("\nExample affected groups:")
print(affected_groups.head(20).to_string(index=False))

Total groups: 14139
Groups with multiple observations: 1909

Example affected groups:
commodity_consolidated                     market pricetype date_month  number_of_rows
                 Beans         Dagahaley (Daadab)    Retail    2023-12               2
                 Beans         Dagahaley (Daadab)    Retail    2024-03               2
                 Beans         Dagahaley (Daadab)    Retail    2024-06               2
                 Beans         Dagahaley (Daadab)    Retail    2024-09               2
                 Beans         Dagahaley (Daadab)    Retail    2025-03               2
                 Beans         Dagahaley (Daadab)    Retail    2025-06               2
                 Beans         Dagahaley (Daadab)    Retail    2025-09               2
                 Beans         Dagahaley (Daadab)    Retail    2025-12               2
                 Beans         Dagahaley (Daadab)    Retail    2026-03               2
                 Beans Eldoret town (Uasin G

In [93]:
# Check which original commodity varieties are being combined
# within the same standardized commodity, market, price type,
# and month.

variety_check = (
    agri_df
    .groupby(consolidation_key)["commodity"]
    .agg(
        number_of_varieties="nunique",
        varieties=lambda x: ", ".join(sorted(x.unique()))
    )
    .reset_index()
)

# Keep only groups where more than one original commodity
# variety is being combined.

variety_check = variety_check[
    variety_check["number_of_varieties"] > 1
].copy()

print(
    "Groups containing multiple commodity varieties:",
    len(variety_check)
)

print("\nExamples:")
print(variety_check.head(20).to_string(index=False))

Groups containing multiple commodity varieties: 1909

Examples:
commodity_consolidated                     market pricetype date_month  number_of_varieties          varieties
                 Beans         Dagahaley (Daadab)    Retail    2023-12                    2 Beans, Beans (dry)
                 Beans         Dagahaley (Daadab)    Retail    2024-03                    2 Beans, Beans (dry)
                 Beans         Dagahaley (Daadab)    Retail    2024-06                    2 Beans, Beans (dry)
                 Beans         Dagahaley (Daadab)    Retail    2024-09                    2 Beans, Beans (dry)
                 Beans         Dagahaley (Daadab)    Retail    2025-03                    2 Beans, Beans (dry)
                 Beans         Dagahaley (Daadab)    Retail    2025-06                    2 Beans, Beans (dry)
                 Beans         Dagahaley (Daadab)    Retail    2025-09                    2 Beans, Beans (dry)
                 Beans         Dagahaley (Daadab

In [94]:
# Check the current AgriPulse columns before aggregation.
# This helps us decide which fields can safely be retained
# and which fields need to be recalculated after consolidation.

print(agri_df.columns.tolist())

['date', 'region', 'county', 'market', 'market_id', 'latitude', 'longitude', 'category', 'commodity', 'commodity_id', 'unit', 'priceflag', 'pricetype', 'currency', 'price', 'usdprice', 'date_month', 'rainfall_mm', 'price_per_kg', 'month', 'year', 'season', 'lag_1', 'lag_3', 'lag_6', 'lag_12', 'rolling_3m_avg', 'rolling_6m_avg', 'rolling_12m_avg', 'price_vs_regional_avg', 'price_label', 'commodity_type', 'commodity_consolidated']


In [95]:
# Define the columns that identify a unique monthly commodity observation.
# Retail and Wholesale remain separate because "pricetype" is included.

consolidation_key = [
    "commodity_consolidated",
    "market",
    "pricetype",
    "date_month"
]

# Columns that describe the observation but do not need to be averaged.
# For these, we keep the first available value within each group.

first_value_columns = [
    "region",
    "county",
    "market_id",
    "latitude",
    "longitude",
    "category",
    "commodity_id",
    "unit",
    "priceflag",
    "currency",
    "month",
    "year",
    "season",
    "commodity_type"
]

# Build the aggregation instructions.
# Price is the important variable: when different varieties occur
# in the same market and month, their price is represented by the median.

aggregation_rules = {
    "date": "first",
    "price_per_kg": "median",
    "rainfall_mm": "median"
}

# Retain the first available value for descriptive fields.
for column in first_value_columns:
    aggregation_rules[column] = "first"

# Perform the consolidation.
agric_df = (
    agri_df
    .groupby(consolidation_key, as_index=False)
    .agg(aggregation_rules)
)

# Rename the consolidated commodity column back to "commodity"
# because this is the final commodity field we want to use.

agric_df = agric_df.rename(
    columns={"commodity_consolidated": "commodity"}
)

print("Consolidated dataset shape:", agric_df.shape)
print("\nColumns:")
print(agric_df.columns.tolist())

Consolidated dataset shape: (14139, 21)

Columns:
['commodity', 'market', 'pricetype', 'date_month', 'date', 'price_per_kg', 'rainfall_mm', 'region', 'county', 'market_id', 'latitude', 'longitude', 'category', 'commodity_id', 'unit', 'priceflag', 'currency', 'month', 'year', 'season', 'commodity_type']


In [96]:
# Define the key that should uniquely identify each monthly
# commodity price observation after consolidation.

final_monthly_key = [
    "commodity",
    "market",
    "pricetype",
    "date_month"
]

# Count duplicate observations using the same key.
# We expect this to be zero after consolidation.

duplicates_after_consolidation = agric_df.duplicated(
    subset=final_monthly_key
).sum()

print(
    "Duplicates after commodity consolidation:",
    duplicates_after_consolidation
)

Duplicates after commodity consolidation: 0


In [97]:
# Identify the FEWS observations that are not already present
# in the AgriPulse dataset.
#
# We compare commodity, market, price type, and month.
# These fields define a unique monthly price observation.

monthly_key = [
    "commodity",
    "market",
    "pricetype",
    "date_month"
]

fews_only = fews_new.merge(
    agric_df[monthly_key].drop_duplicates(),
    on=monthly_key,
    how="left",
    indicator=True
)

# Keep only FEWS observations that do not already exist
# in the consolidated AgriPulse dataset.

fews_only = fews_only[
    fews_only["_merge"] == "left_only"
].copy()

# Remove the temporary merge indicator.

fews_only = fews_only.drop(columns="_merge")

print("FEWS-only observations:", len(fews_only))
print("FEWS-only shape:", fews_only.shape)

FEWS-only observations: 20410
FEWS-only shape: (20410, 11)


In [98]:
# Inspect the columns and a few rows of the FEWS-only observations.
# We need to understand which fields can be directly mapped
# to agric_df and which fields should remain missing.

print("FEWS-only columns:")
print(fews_only.columns.tolist())

print("\nSample FEWS-only observations:")
print(fews_only.head().to_string(index=False))

FEWS-only columns:
['commodity', 'market', 'pricetype', 'date', 'price_per_kg', 'rainfall_mm', 'admin1', 'latitude', 'longitude', 'category', 'date_month']

Sample FEWS-only observations:
  commodity market pricetype       date  price_per_kg  rainfall_mm  admin1  latitude  longitude        category date_month
Beans (dry)  Busia Wholesale 2014-11-01     75.500000       113.30 Western  0.462451    34.1065 pulses and nuts    2014-11
Beans (dry)  Busia Wholesale 2016-03-01     95.000000        65.93 Western  0.462451    34.1065 pulses and nuts    2016-03
Beans (dry)  Busia Wholesale 2016-04-01     90.000000       264.59 Western  0.462451    34.1065 pulses and nuts    2016-04
Beans (dry)  Busia Wholesale 2016-05-01    100.000000       208.40 Western  0.462451    34.1065 pulses and nuts    2016-05
Beans (dry)  Busia Wholesale 2016-06-01     87.222222        84.67 Western  0.462451    34.1065 pulses and nuts    2016-06


In [99]:
# Rename FEWS "admin1" to "region" so it matches the
# corresponding AgriPulse column.

fews_only = fews_only.rename(
    columns={"admin1": "region"}
)

# Standardize FEWS commodity names using the same
# consolidation rules applied to AgriPulse.

fews_only["commodity_consolidated"] = fews_only["commodity"]

for standardized_name, varieties in commodity_groups.items():
    fews_only.loc[
        fews_only["commodity"].isin(varieties),
        "commodity_consolidated"
    ] = standardized_name

# Replace the original commodity column with the
# standardized commodity name.

fews_only["commodity"] = fews_only["commodity_consolidated"]

# Remove the temporary column.

fews_only = fews_only.drop(
    columns=["commodity_consolidated"]
)

# Check the FEWS commodity names after standardization.

print(sorted(fews_only["commodity"].dropna().unique()))

['Beans', 'Maize', 'Maize meal', 'Potatoes', 'Sorghum']


In [100]:
# Get the exact column structure of the consolidated AgriPulse dataset.
# FEWS will be aligned to this structure before integration.

final_columns = agric_df.columns.tolist()

# Add any columns that exist in AgriPulse but are missing from FEWS.
# Missing information will remain as NaN rather than being invented.

for column in final_columns:
    if column not in fews_only.columns:
        fews_only[column] = pd.NA

# Keep the FEWS columns in exactly the same order as AgriPulse.

fews_only = fews_only[final_columns]

# Confirm that both datasets now have the same structure.

print("AgriPulse columns:", len(agric_df.columns))
print("FEWS columns:", len(fews_only.columns))

print("\nColumns match:", list(agric_df.columns) == list(fews_only.columns))

print("\nFEWS-only shape after alignment:", fews_only.shape)

AgriPulse columns: 21
FEWS columns: 21

Columns match: True

FEWS-only shape after alignment: (20410, 21)


In [101]:
# Check whether any FEWS-only observations now match
# an existing AgriPulse observation after commodity standardization.
#
# We use the same monthly identification key:
# commodity + market + price type + month.

overlap_check = fews_only.merge(
    agric_df[final_monthly_key].drop_duplicates(),
    on=final_monthly_key,
    how="inner"
)

print("FEWS observations that now overlap with AgriPulse:", len(overlap_check))

FEWS observations that now overlap with AgriPulse: 1841


In [102]:
# Remove FEWS observations that already exist in the consolidated
# AgriPulse dataset using the standardized monthly identification key.
#
# This prevents duplicate commodity-market-month observations
# from being introduced when we concatenate the datasets.

fews_only = fews_only.merge(
    agric_df[final_monthly_key].drop_duplicates(),
    on=final_monthly_key,
    how="left",
    indicator=True
)

# Keep only observations that are genuinely missing from AgriPulse.
fews_only = fews_only[
    fews_only["_merge"] == "left_only"
].copy()

# Remove the temporary merge indicator.
fews_only = fews_only.drop(columns="_merge")

print("Updated FEWS-only shape:", fews_only.shape)
print("Remaining FEWS-only observations:", len(fews_only))

Updated FEWS-only shape: (18569, 21)
Remaining FEWS-only observations: 18569


In [103]:
# Check whether FEWS-only contains duplicate observations
# using the same key that defines a unique monthly price observation.

fews_duplicates = fews_only.duplicated(
    subset=final_monthly_key
).sum()

print("Duplicates within FEWS-only:", fews_duplicates)

Duplicates within FEWS-only: 0


### consolidated AgriPulse observations with the FEWS observations

In [104]:
# Combine the consolidated AgriPulse observations with the
# FEWS observations that are genuinely missing from AgriPulse.
#
# Both datasets now have the same 21-column structure.

agric_df = pd.concat(
    [agric_df, fews_only],
    ignore_index=True
)

# Check the size of the integrated dataset.
print("Integrated dataset shape:", agric_df.shape)

Integrated dataset shape: (32708, 21)


In [105]:
# Define the key that should uniquely identify each monthly
# commodity price observation.
#
# Retail and Wholesale remain separate through "pricetype".

final_monthly_key = [
    "commodity",
    "market",
    "pricetype",
    "date_month"
]

# Check for duplicate observations using the final key.
duplicates_after_integration = agric_df.duplicated(
    subset=final_monthly_key
).sum()

print(
    "Duplicates after FEWS integration:",
    duplicates_after_integration
)

Duplicates after FEWS integration: 0


In [106]:
# Display the current columns in the fully integrated dataset.
# This helps us confirm which features need to be recreated
# after combining AgriPulse and FEWS.

print(agric_df.columns.tolist())

['commodity', 'market', 'pricetype', 'date_month', 'date', 'price_per_kg', 'rainfall_mm', 'region', 'county', 'market_id', 'latitude', 'longitude', 'category', 'commodity_id', 'unit', 'priceflag', 'currency', 'month', 'year', 'season', 'commodity_type']


In [107]:
# Make sure the date column is in datetime format.
# This ensures we can reliably extract month and year.

agric_df["date"] = pd.to_datetime(agric_df["date"])

# Recreate the month and year from the observation date.
# This overwrites any missing values that came from FEWS.

agric_df["month"] = agric_df["date"].dt.month
agric_df["year"] = agric_df["date"].dt.year

# Check the date range and confirm that the fields were recreated.
print("Date range:", agric_df["date"].min(), "to", agric_df["date"].max())

print("\nSample date features:")
print(
    agric_df[["date", "date_month", "month", "year"]]
    .head()
)

Date range: 2006-01-01 00:00:00 to 2026-08-15 00:00:00

Sample date features:
        date date_month  month  year
0 2024-09-15    2024-09      9  2024
1 2024-03-15    2024-03      3  2024
2 2024-06-15    2024-06      6  2024
3 2024-09-15    2024-09      9  2024
4 2023-12-15    2023-12     12  2023


In [109]:
# Drop columns that are not needed for the integrated
# agricultural commodity price forecasting dataset.
#
# These columns are either unavailable in FEWS or are not
# required for our forecasting analysis.

agric_df = agric_df.drop(
    columns=[
        "priceflag",
        "unit",
        "county"
    ]
)

# Confirm the columns have been removed.
print("Remaining columns:")
print(agric_df.columns.tolist())

print("\nDataset shape:", agric_df.shape)

Remaining columns:
['commodity', 'market', 'pricetype', 'date_month', 'date', 'price_per_kg', 'rainfall_mm', 'region', 'market_id', 'latitude', 'longitude', 'category', 'commodity_id', 'currency', 'month', 'year', 'season', 'commodity_type']

Dataset shape: (32708, 18)


In [110]:
# Check how many missing values remain in each column
# after integrating AgriPulse with FEWS.

missing_values = agric_df.isna().sum()

# Display only columns that contain missing values.
missing_values = missing_values[missing_values > 0]

print("Columns with missing values:")
print(missing_values)

Columns with missing values:
region              106
market_id         18569
latitude            231
longitude           231
category            231
commodity_id      18569
currency          18569
season            18569
commodity_type    18569
dtype: int64


In [111]:
# Identify observations where the region is missing.
# We want to inspect their markets before deciding
# whether the region can be recovered reliably.

missing_region = agric_df[
    agric_df["region"].isna()
].copy()

print("Rows with missing region:", len(missing_region))

print("\nMarkets with missing region:")
print(
    missing_region["market"]
    .value_counts()
    .head(30)
)

Rows with missing region: 106

Markets with missing region:
market
Ugunja            34
Muthurwa Narok    27
Kabati            24
Mandera Town      21
Name: count, dtype: int64


In [112]:
# Find the regions already associated with the markets
# that have missing region values.
#
# We use the existing non-missing records as the reference
# instead of manually assigning regions.

markets_with_missing_region = agric_df.loc[
    agric_df["region"].isna(),
    "market"
].unique()

known_market_regions = (
    agric_df[
        agric_df["market"].isin(markets_with_missing_region)
        & agric_df["region"].notna()
    ]
    .groupby("market")["region"]
    .unique()
)

print("Known regions for the affected markets:")

for market, regions in known_market_regions.items():
    print(f"{market}: {regions}")

Known regions for the affected markets:


In [113]:
# Check whether the observations with missing regions
# have latitude and longitude information.
#
# Coordinates can help us determine the correct region
# without relying on the market name alone.

missing_region_coords = agric_df[
    agric_df["region"].isna()
][
    ["market", "latitude", "longitude"]
]

print(missing_region_coords.groupby("market").agg(
    rows=("market", "size"),
    latitude_values=("latitude", "nunique"),
    longitude_values=("longitude", "nunique"),
    latitude_missing=("latitude", lambda x: x.isna().sum()),
    longitude_missing=("longitude", lambda x: x.isna().sum())
))

                rows  latitude_values  longitude_values  latitude_missing  \
market                                                                      
Kabati            24                1                 1                 0   
Mandera Town      21                1                 1                 0   
Muthurwa Narok    27                1                 1                 0   
Ugunja            34                1                 1                 0   

                longitude_missing  
market                             
Kabati                          0  
Mandera Town                    0  
Muthurwa Narok                  0  
Ugunja                          0  


In [114]:
# Display the unique coordinates for each market with a missing region.
# Since each market has one latitude/longitude pair, we can use
# these coordinates to identify the appropriate region.

market_coordinates = (
    agric_df[
        agric_df["region"].isna()
    ]
    .groupby("market")[["latitude", "longitude"]]
    .first()
    .reset_index()
)

print(market_coordinates.to_string(index=False))

        market  latitude  longitude
        Kabati -0.949560  37.101730
  Mandera Town  3.937300  41.856900
Muthurwa Narok -1.080830  35.871110
        Ugunja  0.180938  34.295511


In [115]:
# Map each affected market to its correct region.
# The assignments are based on the geographic location of
# each market identified from its latitude and longitude.

region_mapping = {
    "Kabati": "Central",
    "Mandera Town": "North Eastern",
    "Muthurwa Narok": "Rift Valley",
    "Ugunja": "Nyanza"
}

# Fill only the missing region values.
# Existing non-missing regions will not be changed.

agric_df["region"] = agric_df["region"].fillna(
    agric_df["market"].map(region_mapping)
)

# Confirm that no region values remain missing.
print("Missing regions after mapping:", agric_df["region"].isna().sum())

Missing regions after mapping: 0


In [116]:
# Identify observations where commodity category is missing.
# We will check which commodities are affected and whether
# their category can be recovered from other observations.

missing_category = agric_df[
    agric_df["category"].isna()
].copy()

print("Rows with missing category:", len(missing_category))

print("\nCommodities with missing category:")
print(
    missing_category["commodity"]
    .value_counts()
)

Rows with missing category: 231

Commodities with missing category:
commodity
Maize meal    231
Name: count, dtype: int64


In [117]:
# Display the existing category labels.
# We want to match the naming convention already used
# in the AgriPulse dataset before filling Maize meal.

print(
    sorted(
        agric_df["category"]
        .dropna()
        .unique()
    )
)

['cereals and tubers', 'pulses and nuts', 'vegetables and fruits']


In [118]:
# Maize meal is a processed maize product, so we assign it
# to the existing "cereals and tubers" category.
#
# Only the missing category values are affected.

agric_df["category"] = agric_df["category"].fillna(
    agric_df["commodity"].eq("Maize meal").map({
        True: "cereals and tubers",
        False: pd.NA
    })
)

# Confirm that all category values are now populated.
print("Missing categories after filling:", agric_df["category"].isna().sum())

Missing categories after filling: 0


In [119]:
# Identify markets with missing latitude or longitude.
# We will check whether those markets have known coordinates
# in other observations in the integrated dataset.

missing_coordinates = agric_df[
    agric_df["latitude"].isna() | agric_df["longitude"].isna()
].copy()

print("Rows with missing coordinates:", len(missing_coordinates))

print("\nMarkets with missing coordinates:")
print(
    missing_coordinates["market"]
    .value_counts()
)

Rows with missing coordinates: 231

Markets with missing coordinates:
market
Nairobi    231
Name: count, dtype: int64


In [120]:
# Find the latitude and longitude values already recorded
# for Nairobi in other observations.

nairobi_coordinates = (
    agric_df[
        (agric_df["market"] == "Nairobi") &
        agric_df["latitude"].notna() &
        agric_df["longitude"].notna()
    ][["latitude", "longitude"]]
    .drop_duplicates()
)

print("Known Nairobi coordinate pairs:")
print(nairobi_coordinates.to_string(index=False))

Known Nairobi coordinate pairs:
 latitude  longitude
 -1.28000    36.8200
 -1.28255    36.8317


In [121]:
# Inspect which commodities and price types are associated with
# each of the two existing Nairobi coordinate pairs.
#
# This helps us determine whether they represent different
# market locations or simply minor coordinate differences.

nairobi_location_check = (
    agric_df[
        (agric_df["market"] == "Nairobi") &
        agric_df["latitude"].notna() &
        agric_df["longitude"].notna()
    ]
    .groupby(["latitude", "longitude"])
    .agg(
        rows=("market", "size"),
        commodities=("commodity", lambda x: ", ".join(sorted(x.unique()))),
        price_types=("pricetype", lambda x: ", ".join(sorted(x.unique())))
    )
    .reset_index()
)

print(nairobi_location_check.to_string(index=False))

 latitude  longitude  rows                                             commodities       price_types
 -1.28255    36.8317   143                                   Beans, Maize, Sorghum         Wholesale
 -1.28000    36.8200   715 Beans, Maize, Onions, Potatoes, Rice, Sorghum, Tomatoes Retail, Wholesale


In [122]:
# Inspect the characteristics of the Nairobi observations
# that are missing coordinates.
#
# We will compare these with the two known Nairobi locations
# to determine whether the correct coordinate can be inferred.

missing_nairobi = agric_df[
    (agric_df["market"] == "Nairobi") &
    (agric_df["latitude"].isna() | agric_df["longitude"].isna())
].copy()

print(
    missing_nairobi[
        ["commodity", "pricetype", "date", "price_per_kg"]
    ]
    .head(30)
    .to_string(index=False)
)

 commodity pricetype       date  price_per_kg
Maize meal    Retail 2006-01-31        25.345
Maize meal    Retail 2006-02-28        26.285
Maize meal    Retail 2006-03-31        26.240
Maize meal    Retail 2006-04-30        26.805
Maize meal    Retail 2006-05-31        27.955
Maize meal    Retail 2006-06-30        28.400
Maize meal    Retail 2006-07-31        28.580
Maize meal    Retail 2006-08-31        28.460
Maize meal    Retail 2006-09-30        27.885
Maize meal    Retail 2006-10-31        27.485
Maize meal    Retail 2006-11-30        26.540
Maize meal    Retail 2006-12-31        26.220
Maize meal    Retail 2007-01-31        25.925
Maize meal    Retail 2007-02-28        24.840
Maize meal    Retail 2007-03-31        24.315
Maize meal    Retail 2007-04-30        23.775
Maize meal    Retail 2007-05-31        23.705
Maize meal    Retail 2007-06-30        23.835
Maize meal    Retail 2007-07-31        23.820
Maize meal    Retail 2007-08-31        23.900
Maize meal    Retail 2007-09-30   

In [123]:
# Confirm that all remaining missing coordinates belong to
# the FEWS Maize meal observations.

missing_coordinate_check = agric_df[
    agric_df["latitude"].isna() | agric_df["longitude"].isna()
]

print(
    missing_coordinate_check[
        ["commodity", "market", "pricetype"]
    ]
    .drop_duplicates()
    .to_string(index=False)
)

 commodity  market pricetype
Maize meal Nairobi    Retail


In [124]:
# Check whether each market in the original AgriPulse data
# consistently maps to a single market_id.
#
# If every market has one unique ID, we can potentially
# recover market IDs for the FEWS observations.

market_id_check = (
    agric_df[
        agric_df["market_id"].notna()
    ]
    .groupby("market")["market_id"]
    .nunique()
    .reset_index(name="unique_market_ids")
)

print("Markets with more than one market_id:")

print(
    market_id_check[
        market_id_check["unique_market_ids"] > 1
    ]
    .to_string(index=False)
)

print(
    "\nTotal markets with multiple IDs:",
    (market_id_check["unique_market_ids"] > 1).sum()
)

Markets with more than one market_id:
Empty DataFrame
Columns: [market, unique_market_ids]
Index: []

Total markets with multiple IDs: 0


In [125]:
# Create a lookup showing which AgriPulse market IDs correspond
# to each market name.

market_id_lookup = (
    agric_df[
        agric_df["market_id"].notna()
    ][["market", "market_id"]]
    .drop_duplicates()
)

# Check how many FEWS-only rows have a market that already
# exists in the AgriPulse market list.

fews_market_match = fews_only["market"].isin(
    market_id_lookup["market"]
)

print(
    "FEWS-only rows with a known AgriPulse market:",
    fews_market_match.sum()
)

print(
    "FEWS-only rows with a new market:",
    (~fews_market_match).sum()
)


FEWS-only rows with a known AgriPulse market: 2533
FEWS-only rows with a new market: 16036


In [126]:
# Create a market-to-market_id lookup from the existing AgriPulse data.
# Each market has been verified to have only one market_id.

market_id_lookup = (
    agric_df[
        agric_df["market_id"].notna()
    ][["market", "market_id"]]
    .drop_duplicates("market")
    .set_index("market")["market_id"]
)

# Fill market_id only where it is currently missing.
# Markets that are new to AgriPulse will remain missing.

agric_df["market_id"] = agric_df["market_id"].fillna(
    agric_df["market"].map(market_id_lookup)
)

# Check how many market IDs are still missing.
print(
    "Missing market IDs after mapping:",
    agric_df["market_id"].isna().sum()
)

Missing market IDs after mapping: 16036


In [127]:
# Check whether each standardized commodity maps to a single
# commodity_id in the existing AgriPulse data.
#
# This is important because several original varieties were
# consolidated into one commodity name.

commodity_id_check = (
    agric_df[
        agric_df["commodity_id"].notna()
    ]
    .groupby("commodity")["commodity_id"]
    .nunique()
    .reset_index(name="unique_commodity_ids")
)

print("Standardized commodities with multiple commodity IDs:")

print(
    commodity_id_check[
        commodity_id_check["unique_commodity_ids"] > 1
    ]
    .to_string(index=False)
)

print(
    "\nTotal commodities with multiple IDs:",
    (commodity_id_check["unique_commodity_ids"] > 1).sum()
)

Standardized commodities with multiple commodity IDs:
commodity  unique_commodity_ids
    Beans                     7
    Maize                     3
   Onions                     2
 Potatoes                     3
  Sorghum                     3

Total commodities with multiple IDs: 5


In [128]:
# Drop commodity_id because the original IDs refer to individual
# commodity varieties that were consolidated into broader groups.
#
# Keeping one ID for a consolidated commodity would be misleading
# because Beans, Maize, Onions, Potatoes, and Sorghum each came
# from multiple original commodity IDs.

agric_df = agric_df.drop(
    columns=["commodity_id"]
)

# Confirm that commodity_id has been removed.

print("Remaining columns:")
print(agric_df.columns.tolist())

print("\nDataset shape:", agric_df.shape)

Remaining columns:
['commodity', 'market', 'pricetype', 'date_month', 'date', 'price_per_kg', 'rainfall_mm', 'region', 'market_id', 'latitude', 'longitude', 'category', 'currency', 'month', 'year', 'season', 'commodity_type']

Dataset shape: (32708, 17)


In [129]:
# Check the currency values available in the integrated dataset.
# We want to see whether AgriPulse uses one consistent currency
# before deciding how to handle the missing FEWS values.

print("Currency values:")
print(agric_df["currency"].value_counts(dropna=False))

Currency values:
currency
NaN    18569
KES    14139
Name: count, dtype: int64


In [130]:
# FEWS prices are being integrated into the same Kenyan price dataset.
# Standardize the missing currency values to KES, matching the AgriPulse data.

agric_df["currency"] = agric_df["currency"].fillna("KES")

# Confirm the result
print("Currency values after filling:")
print(agric_df["currency"].value_counts(dropna=False))

Currency values after filling:
currency
KES    32708
Name: count, dtype: int64


In [131]:
# Check which season labels exist in the integrated dataset.
# We also compare seasons against months to understand the original
# AgriPulse season definitions before filling the FEWS values.

print("Season values:")
print(agric_df["season"].value_counts(dropna=False))

print("\nSeason by month:")
print(
    pd.crosstab(
        agric_df["month"],
        agric_df["season"],
        dropna=False
    )
)

Season values:
season
NaN            18569
Dry Season      4902
Long Rains      3970
Short Rains     3247
Cool Dry        2020
Name: count, dtype: int64

Season by month:
season  Cool Dry  Dry Season  Long Rains  Short Rains   NaN
month                                                      
1            965           0           0            0  1782
2           1055           0           0            0  1770
3              0           0        1990            0  1755
4              0           0         963            0  1698
5              0           0        1017            0  1679
6              0        1524           0            0  1677
7              0        1058           0            0  1548
8              0         897           0            0  1450
9              0        1423           0            0  1447
10             0           0           0          896  1432
11             0           0           0          849  1220
12             0           0           0         

In [132]:
# Reconstruct season from month using the season pattern
# observed in the original AgriPulse data.

season_mapping = {
    1: "Cool Dry",
    2: "Cool Dry",
    3: "Long Rains",
    4: "Long Rains",
    5: "Long Rains",
    6: "Dry Season",
    7: "Dry Season",
    8: "Dry Season",
    9: "Dry Season",
    10: "Short Rains",
    11: "Short Rains",
    12: "Short Rains"
}

# Fill only the missing season values.
agric_df["season"] = agric_df["season"].fillna(
    agric_df["month"].map(season_mapping)
)

# Confirm that no season values remain missing.
print("Missing season values:", agric_df["season"].isna().sum())

print("\nSeason values after filling:")
print(agric_df["season"].value_counts())

Missing season values: 0

Season values after filling:
season
Dry Season     11024
Long Rains      9102
Short Rains     7010
Cool Dry        5572
Name: count, dtype: int64


In [81]:
# Preserve the original commodity name before consolidation.
# This allows us to trace an aggregated observation back to
# the commodity varieties that contributed to it.

agric_df["commodity_original"] = agric_df["commodity"]

In [82]:
# Create a mapping from each original commodity variety
# to its consolidated commodity name.

commodity_mapping = {
    original: consolidated
    for consolidated, varieties in commodity_groups.items()
    for original in varieties
}

# Replace the original commodity names with the consolidated names.
agric_df["commodity"] = agric_df["commodity"].replace(commodity_mapping)

In [83]:
# Group observations that now represent the same commodity,
# market, price type, and month.
#
# The median price is used to represent multiple observations
# within the same group and reduce the influence of unusually
# high or low prices.

grouping_columns = [
    "commodity",
    "market",
    "pricetype",
    "date_month"
]

agric_df = (
    agric_df
    .groupby(grouping_columns, as_index=False)
    .agg(
        price_per_kg=("price_per_kg", "median")
    )
)

In [84]:
# Check the structure of the consolidated dataset.
# This confirms which columns remain after the groupby aggregation.

print("Dataset shape:", agric_df.shape)

print("\nColumns:")
print(agric_df.columns.tolist())

Dataset shape: (32708, 5)

Columns:
['commodity', 'market', 'pricetype', 'date_month', 'price_per_kg']


In [86]:
# Define the columns that identify an observation at the
# commodity-market-price-type-month level.
monthly_key = [
    "commodity",
    "market",
    "pricetype",
    "date_month"
]

print("AgriPulse shape:", agri_df.shape)
print("FEWS shape:", fews_new.shape)

AgriPulse shape: (16327, 32)
FEWS shape: (22049, 11)


In [133]:
# Check the commodity types currently assigned to each commodity.
# This helps us identify whether the existing AgriPulse mappings
# can be safely used to fill the missing FEWS values.

commodity_type_check = (
    agric_df
    .groupby("commodity")["commodity_type"]
    .value_counts(dropna=False)
    .unstack(fill_value=0)
)

print(commodity_type_check)

commodity_type     Cereal  Legume  Tuber/Root Crop  Vegetable    NaN
commodity                                                           
Beans                   0    2886                0          0   7069
Cabbage                 0       0                0        450      0
Cowpea leaves           0       0                0         40      0
Cowpeas                 0     255                0          0      0
Cowpeas (dry)           0     292                0          0      0
Kale                    0       0                0        803      0
Maize                3361       0                0          0  10425
Maize meal              0       0                0          0    231
Millet (finger)        77       0                0          0      0
Onions                  0       0                0        470      0
Pigeon peas (dry)       0     111                0          0      0
Potatoes                0       0             1740          0    123
Rice                  746       0 

In [134]:
# Assign commodity types to the standardized commodities.
# These mappings are based on the existing AgriPulse classifications.

commodity_type_mapping = {
    "Beans": "Legume",
    "Maize": "Cereal",
    "Maize meal": "Cereal",
    "Millet (finger)": "Cereal",
    "Rice": "Cereal",
    "Rice (aromatic)": "Cereal",
    "Sorghum": "Cereal",
    "Cowpeas": "Legume",
    "Cowpeas (dry)": "Legume",
    "Pigeon peas (dry)": "Legume",
    "Potatoes": "Tuber/Root Crop",
    "Cabbage": "Vegetable",
    "Cowpea leaves": "Vegetable",
    "Kale": "Vegetable",
    "Onions": "Vegetable",
    "Spinach": "Vegetable",
    "Tomatoes": "Vegetable"
}

# Fill only the missing commodity_type values.
agric_df["commodity_type"] = agric_df["commodity_type"].fillna(
    agric_df["commodity"].map(commodity_type_mapping)
)

# Confirm the result.
print("Missing commodity_type values:", agric_df["commodity_type"].isna().sum())

print("\nCommodity type values:")
print(agric_df["commodity_type"].value_counts())

Missing commodity_type values: 0

Commodity type values:
commodity_type
Cereal             17687
Legume             10613
Vegetable           2545
Tuber/Root Crop     1863
Name: count, dtype: int64


In [135]:
# Perform a final missing-value audit after the integration and
# metadata cleaning steps.

missing_values = (
    agric_df.isna()
    .sum()
    .sort_values(ascending=False)
)

print("Missing values by column:")
print(missing_values[missing_values > 0])

Missing values by column:
market_id    16036
latitude       231
longitude      231
dtype: int64


In [136]:
# Check that each commodity-market-price-type-month combination
# appears only once after integration and cleaning.

final_key = [
    "commodity",
    "market",
    "pricetype",
    "date_month"
]

duplicate_count = agric_df.duplicated(
    subset=final_key
).sum()

print("Duplicate observations:", duplicate_count)

Duplicate observations: 0


In [137]:
agripulse_final = agric_df.copy()

In [138]:
# Save the final integrated dataset as a CSV file.
# index=False prevents pandas from adding an unnecessary index column.

agripulse_final.to_csv(
    "agripulse_kenya_final.csv",
    index=False
)

print("Dataset saved successfully as agripulse_kenya_final.csv")

Dataset saved successfully as agripulse_kenya_final.csv
